## Cell 1 — Import libraries

### Theory: Why these libraries are required
- **`logging`** records operational events such as request arrival, successful predictions, and exceptions. Unlike `print()`, logs can be timestamped, assigned severity levels, and saved to a file.
- **`time`** provides high-resolution timing so prediction latency can be measured.
- **`Path`** creates and manages file-system paths in a platform-independent way.
- **pandas** loads the Kidney Disease CSV and represents tabular input as a DataFrame.
- **scikit-learn utilities** split data, preprocess missing/categorical values, train a Random Forest, and evaluate classification performance.

This practical demonstrates **application logging and error tracking** around a small ML prediction function. It is a classroom example; the dataset/model is not intended for clinical decision-making.

In [1]:
# Import the standard logging module for timestamped operational events.
import logging
# Use a high-resolution timer to measure prediction latency.
import time
# Use Path for portable directory and file handling.
from pathlib import Path

# pandas is used to load and manipulate the CSV dataset.
import pandas as pd

# Split data into training and test subsets.
from sklearn.model_selection import train_test_split
# Apply different preprocessing to numeric and categorical columns.
from sklearn.compose import ColumnTransformer
# Chain preprocessing and the estimator into one reusable workflow.
from sklearn.pipeline import Pipeline
# Fill missing values in feature columns.
from sklearn.impute import SimpleImputer
# Convert categorical values into numeric indicator columns.
from sklearn.preprocessing import OneHotEncoder
# Use a tree-ensemble classifier for this baseline experiment.
from sklearn.ensemble import RandomForestClassifier
# Calculate classification performance metrics.
from sklearn.metrics import accuracy_score, classification_report

print("All required libraries imported successfully.")

All required libraries imported successfully.


## Cell 2 — Load and inspect the dataset

### Theory: Data inspection before model use
Before training or monitoring a model, inspect the dataset's dimensions, column names, sample records, and target distribution. This can reveal misspelled columns, whitespace in labels, unexpected missing values, and class imbalance.

The notebook expects `kidney_disease_cleaned.csv` in the current working directory. The `classification` column is the prediction target. Whitespace is trimmed from column names and text fields to reduce avoidable inconsistencies.

In [2]:
# Path to the cleaned Kidney Disease dataset; keep the CSV in the notebook's working directory.
DATASET_PATH = "kidney_disease_cleaned.csv"

# Read the CSV into a pandas DataFrame.
df = pd.read_csv(DATASET_PATH)
# Remove accidental spaces around column names.
df.columns = df.columns.str.strip()

# Clean whitespace in text columns
# Clean whitespace in text columns
# Trim leading/trailing spaces in object/string columns.
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

# Report row and column counts to confirm the data loaded as expected.
print("Dataset shape:", df.shape)
# Show exact column names so schema problems are easy to spot.
print("\nColumn names:")
print(df.columns.tolist())

# Preview a few records to inspect values and formatting.
print("\nFirst five records:")
display(df.head())

# Inspect class frequencies, including missing target labels.
print("\nTarget distribution:")
print(df["classification"].value_counts(dropna=False))

Dataset shape: (400, 26)

Column names:
['id', 'age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'classification']

First five records:


,id,age,bp,sg,al,su,rbc,pc,pcc,ba,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,...,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,...,38,6000,NaN,no,no,no,good,no,no,ckd
2,2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,...,31,7500,NaN,no,yes,no,poor,no,yes,ckd
3,3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,...,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,...,35,7300,4.6,no,no,no,good,no,no,ckd



Target distribution:
classification
ckd       250
notckd    150
Name: count, dtype: int64


## Cell 3 — Prepare features, preprocess data, and train a baseline model

### Theory: Establishing a baseline
A baseline is a reference point against which later production behavior can be assessed. Here, the target labels are normalized, unsupported/missing labels are removed, and the identifier (`id`) is excluded from the model features.

- **Train/test split:** the test set estimates performance on records not used for fitting. `stratify=y` preserves class proportions where possible.
- **ColumnTransformer:** applies different preprocessing to numeric and categorical columns.
- **Median imputation:** fills missing numeric values using the median.
- **Most-frequent imputation + one-hot encoding:** fills missing categorical values and converts categories into numeric indicator columns. `handle_unknown="ignore"` lets the encoder handle previously unseen categories without failing.
- **Pipeline:** ensures the same preprocessing steps are applied during training and prediction.
- **Random Forest:** provides a baseline classifier. `class_weight="balanced"` adjusts class weights based on their frequency.

The printed accuracy and classification report are baseline metrics, not proof that the model is suitable for medical use.

In [3]:
# Normalize target labels to reduce variations caused by case, spaces, or a trailing period.
target = (
    df["classification"]
    .astype("string")
    .str.lower()
    .str.strip()
    .str.replace(r"\.$", "", regex=True)
    .str.replace(r"\s+", "", regex=True)
)

# Map recognized target values to consistent class labels.
label_map = {
    "ckd": "ckd",
    "notckd": "notckd",
    "not_ckd": "notckd"
}

df["classification"] = target.map(label_map)

# Remove records with missing or unsupported target labels
# Remove rows whose target is missing or not recognized by the label map.
df = df.dropna(subset=["classification"]).copy()

# Remove identifier and target from input features
# Separate input features from the label; ignore the ID column if it exists.
X = df.drop(columns=["classification", "id"], errors="ignore")
# Store the target labels separately for supervised learning.
y = df["classification"]

# Stop early if cleaning left fewer than two classes.
if y.nunique() < 2:
    raise ValueError(
        "At least two target classes are required. "
        "Inspect the classification labels."
    )

# Separate numerical and categorical features
# Identify numeric columns for numeric-specific preprocessing.
numeric_features = X.select_dtypes(
    include="number"
).columns.tolist()

# Identify non-numeric columns for categorical preprocessing.
categorical_features = X.select_dtypes(
    exclude="number"
).columns.tolist()

# Preprocessing pipeline
# Define a transformer that imputes numeric and categorical missing values appropriately.
preprocessor = ColumnTransformer([
    (
        "numeric",
# Median imputation is robust to skew and outliers compared with mean imputation.
        SimpleImputer(strategy="median"),
        numeric_features
    ),
    (
        "categorical",
        Pipeline([
# Fill missing categories with the most frequent value, then one-hot encode categories.
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_features
    )
])

# Model pipeline
# Assemble preprocessing and classifier into one pipeline so prediction repeats training transforms.
model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "classifier",
# Random Forest uses 100 trees, reproducible randomness, and class-balanced weights.
        RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            class_weight="balanced"
        )
    )
])

# Hold out 20% for evaluation and preserve class proportions in each split.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Fit preprocessing and the classifier using training data only.
model.fit(X_train, y_train)

# Generate predictions for held-out examples.
baseline_predictions = model.predict(X_test)

# Report split sizes and baseline test accuracy.
print("Model training completed.")
print("Training records:", len(X_train))
print("Test records:", len(X_test))
print("Test accuracy:", round(
    accuracy_score(y_test, baseline_predictions), 4
))
# Show precision, recall, F1-score, and support for each class.
print("\nClassification report:")
print(classification_report(
    y_test,
    baseline_predictions,
    zero_division=0
))

Model training completed.
Training records: 320
Test records: 80
Test accuracy: 0.9875

Classification report:
              precision    recall  f1-score   support

         ckd       0.98      1.00      0.99        50
      notckd       1.00      0.97      0.98        30

    accuracy                           0.99        80
   macro avg       0.99      0.98      0.99        80
weighted avg       0.99      0.99      0.99        80



## Cell 4 — Configure file-based logging

### Theory: Log levels and persistent logs
A log record commonly includes a timestamp, severity level, logger name, and message. This notebook writes INFO-level events to `logs/ml_monitor.log`.

The log directory is created if needed. Existing root handlers are cleared and `force=True` resets logging configuration, helping avoid duplicate output when rerunning cells in a notebook. In larger applications, logging configuration is normally centralized and carefully managed rather than repeatedly reset.

In [4]:
# Create a directory for log files; parents=True avoids errors if parent folders are missing.
LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Choose the persistent log file location.
LOG_FILE = LOG_DIR / "ml_monitor.log"

# Reset existing handlers so notebook reruns do not duplicate logs
# Reset existing handlers so notebook reruns do not duplicate logs
# Remove and close handlers from previous notebook executions.
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
    handler.close()

# Configure INFO-level logs with timestamp, level, logger name, and message.
logging.basicConfig(
    filename=str(LOG_FILE),
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    force=True
)

# Create a named logger for this ML prediction demo.
logger = logging.getLogger("kidney_ml")

# Record that logging is ready and model preparation has completed.
logger.info("Logging system initialized")
logger.info("Dataset prepared and model training completed")

print("Logging configured successfully.")
print("Log file:", LOG_FILE.resolve())

Logging configured successfully.
Log file: C:\Users\Admin\logs\ml_monitor.log


## Cell 5 — Build a prediction function with logging and error handling

### Theory: Instrumenting an ML service
A production prediction endpoint should make requests observable. This function records:
- a unique request identifier, useful for correlating related log entries;
- the arrival of a prediction request;
- validation failures, including wrong input type, empty input, or missing required features;
- successful prediction count and latency in milliseconds;
- full exception details for failures.

`time.perf_counter()` is intended for measuring elapsed time. The `try/except` block logs the exception and then re-raises it so callers still receive the error. Extra input columns are ignored and columns are reordered to match the training features.

This is a local function, not a deployed API. In production, logs should avoid sensitive patient information and should be protected, rotated, and retained according to organizational policy.

In [5]:
# Wrap model inference with request tracking, validation, timing, and exception logging.
def predict_with_logging(input_data):
# Create a request ID so messages from this call can be correlated.
    request_id = f"REQ-{time.time_ns()}"
# Start a high-resolution timer before validation and prediction.
    start_time = time.perf_counter()

# Log request arrival without logging potentially sensitive input values.
    logger.info(
        "Prediction request received | request_id=%s",
        request_id
    )

    try:
        # Validate the input object
# Validate that the caller supplied the expected tabular data type.
        if not isinstance(input_data, pd.DataFrame):
            raise TypeError(
                "Input must be a pandas DataFrame"
            )

# Reject zero-row requests before model inference.
        if input_data.empty:
            raise ValueError("Input data is empty")

        # Check required features
# Find training features that are absent from the incoming DataFrame.
        missing_features = sorted(
            set(X.columns) - set(input_data.columns)
        )

# Reject inputs that do not contain every feature used during training.
        if missing_features:
            raise ValueError(
                f"Missing required features: {missing_features}"
            )

        # Extra columns are ignored; expected order is enforced
# Reorder columns to the training schema and ignore unexpected extra columns.
        input_data = input_data.loc[:, X.columns]

        # Run model prediction
# Run the trained preprocessing-and-classification pipeline.
        predictions = model.predict(input_data)

# Convert elapsed seconds into milliseconds for readable operational monitoring.
        latency_ms = (
            time.perf_counter() - start_time
        ) * 1000

# Log request arrival without logging potentially sensitive input values.
        logger.info(
            "Prediction successful | request_id=%s | "
            "records=%d | latency_ms=%.2f",
            request_id,
            len(input_data),
            latency_ms
        )

# Return predictions to the caller.
        return predictions

# Log the exception traceback and request ID, then let the caller handle the error.
    except Exception:
        logger.exception(
            "Prediction failed | request_id=%s",
            request_id
        )
        raise

## Cell 6 — Test a valid single-record prediction

### Theory: Happy-path testing
A **happy path** is a normal, valid request. Selecting one row with `iloc[[0]]` keeps the input as a DataFrame (rather than a Series), matching the prediction function's input contract.

The function should log request receipt and successful completion, including the latency and number of records. The printed class is the model's predicted label. Exact predictions and timing can vary with dataset contents and execution environment.

In [6]:
# Select one held-out row; double brackets preserve the DataFrame type.
sample = X_test.iloc[[0]]

# Try a normal request and show the predicted class if it succeeds.
try:
    prediction = predict_with_logging(sample)

    print("Prediction successful.")
    print("Predicted class:", prediction[0])

# Catch unexpected failures so the notebook can continue running.
except Exception as exc:
    print("Prediction failed:", type(exc).__name__, str(exc))

Prediction successful.
Predicted class: ckd


## Cell 7 — Test a request with a missing feature

### Theory: Input validation and error tracking
ML models generally expect the same feature schema used during training. This cell deliberately removes one required column. The prediction function should detect the missing feature before calling the model, log the failure with exception details, and raise a `ValueError`.

This is a controlled negative test: an error is expected, and the `try/except` block keeps the notebook running so subsequent tests can continue.

In [7]:
# Copy one valid record so the test does not modify X_test.
invalid_sample = X_test.iloc[[0]].copy()

# Remove one feature required by the model
# Remove one required feature to simulate a malformed request.
feature_to_remove = X.columns[0]
invalid_sample = invalid_sample.drop(
    columns=[feature_to_remove]
)

# Confirm which feature was removed.
print("Removed feature:", feature_to_remove)

# The function should reject this request and write an error log.
try:
    predict_with_logging(invalid_sample)

# Display the expected validation error without stopping later cells.
except Exception as exc:
    print("Expected error captured.")
    print("Error type:", type(exc).__name__)
    print("Error message:", str(exc))

Removed feature: age
Expected error captured.
Error type: ValueError
Error message: Missing required features: ['age']


## Cell 8 — Test an empty DataFrame

### Theory: Boundary-condition testing
An empty DataFrame has the correct general data type but contains zero records. It should not be passed to the model as a normal prediction request.

This cell verifies that the function rejects empty input with a clear validation error and records the failure. Boundary tests like this help prevent confusing downstream exceptions and make failures easier to diagnose.

In [8]:
# An empty slice has zero rows and should be rejected by the validation function.
try:
    predict_with_logging(X_test.iloc[0:0])

# Report the expected error type and message.
except Exception as exc:
    print("Empty-input test:", type(exc).__name__, str(exc))

Empty-input test: ValueError Input data is empty


## Cell 9 — Test the wrong input type

### Theory: Defensive programming
The prediction function's contract requires a pandas DataFrame. Passing a string deliberately violates that contract. The function should raise a `TypeError`, log the exception, and re-raise it.

Checking input types early gives developers a clearer error than allowing an unrelated model or preprocessing component to fail later.

In [9]:
# Pass a string instead of a DataFrame to test input-type validation.
try:
    predict_with_logging("invalid input")

# Capture and display the expected TypeError.
except Exception as exc:
    print("Wrong-type test:", type(exc).__name__, str(exc))

Wrong-type test: TypeError Input must be a pandas DataFrame


## Cell 10 — Test batch prediction

### Theory: Batch inference
Batch inference sends multiple records through the model in one call. This cell selects up to ten test rows and verifies that the number of predictions corresponds to the number of input records.

Batch processing is useful for throughput-oriented workloads, but latency should be interpreted alongside batch size: a ten-row batch's total request latency is not directly equivalent to the latency of one single-record request.

In [10]:
# Select up to ten test records to demonstrate batch inference.
batch = X_test.iloc[:10]

# Run a batch request and report how many predictions were returned.
try:
    predictions = predict_with_logging(batch)

    print("Records predicted:", len(predictions))
    print("Predictions:", predictions)

# Report batch-level errors if inference fails.
except Exception as exc:
    print("Batch prediction failed:", str(exc))

Records predicted: 10
Predictions: ['ckd' 'ckd' 'notckd' 'notckd' 'notckd' 'ckd' 'ckd' 'ckd' 'notckd' 'ckd']


## Cell 11 — Read the log file

### Theory: Inspecting operational evidence
After successful and intentionally failing requests, the log file provides a persistent record of what happened. Reading the file makes it possible to check timestamps, request IDs, success messages, and exception traces.

Logs are valuable for debugging, auditing, and operational monitoring. They are not automatically a metrics dashboard; later cells parse selected log messages into summary statistics.

In [11]:
# Open the log file as UTF-8 text and read all recorded events.
with open(LOG_FILE, "r", encoding="utf-8") as file:
    log_contents = file.read()

# Print the raw logs so request and error events can be inspected.
print(log_contents)

2026-10-14 16:41:37 | INFO | kidney_ml | Logging system initialized
2026-10-14 16:41:37 | INFO | kidney_ml | Dataset prepared and model training completed
2026-10-14 16:45:05 | INFO | kidney_ml | Prediction request received | request_id=REQ-1791976505325971800
2026-10-14 16:45:05 | INFO | kidney_ml | Prediction successful | request_id=REQ-1791976505325971800 | records=1 | latency_ms=18.86
2026-10-14 16:45:31 | INFO | kidney_ml | Prediction request received | request_id=REQ-1791976531726306000
2026-10-14 16:45:31 | ERROR | kidney_ml | Prediction failed | request_id=REQ-1791976531726306000
Traceback (most recent call last):
  File "C:\Users\Admin\AppData\Local\Temp\ipykernel_3792\3118333893.py", line 26, in predict_with_logging
    raise ValueError(
        f"Missing required features: {missing_features}"
    )
ValueError: Missing required features: ['age']
2026-10-14 16:46:58 | INFO | kidney_ml | Prediction request received | request_id=REQ-1791976618132507400
2026-10-14 16:46:58 | ERRO

## Cell 12 — Summarize request success and failure

### Theory: Turning logs into operational metrics
This cell separates log lines into request-received, successful, and failed events, then computes success and failure rates.

The rates use the number of logged request-received events as the denominator. They are meaningful only if each request is represented consistently in the log. In this simple example, a failed request has both a “request received” line and a “prediction failed” line. For more robust production monitoring, structured logs or dedicated counters are safer than matching human-readable text.

The summary is displayed as a one-row DataFrame for easy review and CSV export.

In [12]:
# Read each log record as a separate line.
log_lines = LOG_FILE.read_text(
    encoding="utf-8"
).splitlines()

# Keep only lines indicating that a prediction request arrived.
prediction_requests = [
    line for line in log_lines
    if "Prediction request received" in line
]

# Keep only successful prediction records; these include latency fields.
successful_requests = [
    line for line in log_lines
    if "Prediction successful" in line
]

# Keep only failed prediction records.
failed_requests = [
    line for line in log_lines
    if "Prediction failed" in line
]

# Count the three event types.
total_requests = len(prediction_requests)
success_count = len(successful_requests)
failure_count = len(failed_requests)

# Calculate percentages safely when there are no request events.
success_rate = (
    success_count / total_requests * 100
    if total_requests else 0
)

# Calculate the failure percentage using the same request denominator.
failure_rate = (
    failure_count / total_requests * 100
    if total_requests else 0
)

# Package the operational counts and rates into a one-row table.
summary = pd.DataFrame([{
    "total_requests": total_requests,
    "successful_requests": success_count,
    "failed_requests": failure_count,
    "success_rate_percent": round(success_rate, 2),
    "failure_rate_percent": round(failure_rate, 2)
}])

# Display the summary for review.
display(summary)

,total_requests,successful_requests,failed_requests,success_rate_percent,failure_rate_percent
0,5,2,3,40.0,60.0


## Cell 13 — Extract prediction latency statistics

### Theory: Latency monitoring
Latency is the elapsed time between starting a request and completing its prediction. The successful log messages include a `latency_ms` field, which this cell extracts using a regular expression.

The notebook calculates the number of successful requests with a recorded latency, plus average, minimum, and maximum latency. These are simple descriptive statistics. For production systems, percentiles such as p50, p95, and p99 are often more informative because averages can hide slow outliers. The timings here are local demonstration measurements, not service-level guarantees.

In [13]:
# Regular expressions are used to extract numeric latency values from log text.
import re

# Store parsed latency values in milliseconds.
latency_values = []

# Inspect successful log messages because only those include recorded latency.
for line in successful_requests:
# Match the numeric portion after latency_ms=.
    match = re.search(r"latency_ms=([\d.]+)", line)

# Convert the captured text into a floating-point number.
    if match:
        latency_values.append(float(match.group(1)))

# Calculate simple descriptive statistics when at least one latency was found.
if latency_values:
# Display the number of measurements and average/minimum/maximum latency.
    latency_summary = pd.DataFrame([{
        "successful_requests_with_latency": len(latency_values),
        "average_latency_ms": round(
            sum(latency_values) / len(latency_values), 3
        ),
        "minimum_latency_ms": round(min(latency_values), 3),
        "maximum_latency_ms": round(max(latency_values), 3)
    }])

    display(latency_summary)

else:
# Explain why no table is shown if no successful latency values were parsed.
    print("No successful prediction latency values found.")

,successful_requests_with_latency,average_latency_ms,minimum_latency_ms,maximum_latency_ms
0,2,18.525,18.19,18.86


## Cell 14 — Save monitoring reports

### Theory: Persisting results
This cell writes the request summary and, when latency data exists, the latency summary to CSV files in `monitoring_reports/`. Saving reports makes the results available after the notebook session ends and allows students to inspect them with spreadsheet software or reuse them in later analysis.

The code creates the directory if it does not already exist. If no successful latency values were extracted, the latency report is not written.

In [14]:
# Create a folder for exported monitoring reports.
REPORT_DIR = Path("monitoring_reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Save request counts and success/failure rates as CSV.
summary.to_csv(
    REPORT_DIR / "logging_summary.csv",
    index=False
)

# Save latency statistics only when measurements exist.
if latency_values:
    latency_summary.to_csv(
        REPORT_DIR / "prediction_latency_summary.csv",
        index=False
    )

# Confirm where report files were saved.
print("Reports saved successfully:")
print(REPORT_DIR.resolve())
print(" - logging_summary.csv")

# Save latency statistics only when measurements exist.
if latency_values:
    print(" - prediction_latency_summary.csv")

Reports saved successfully:
C:\Users\Admin\monitoring_reports
 - logging_summary.csv
 - prediction_latency_summary.csv


## Cell 15 — Practical review and limitations

### What this notebook demonstrates
1. Configure a persistent log file.
2. Record prediction request IDs, success/failure events, and latency.
3. Validate input type, emptiness, and required feature columns.
4. Exercise valid requests and expected error cases.
5. Convert logs into request-rate and latency summaries.
6. Export summary reports as CSV.

### Important limitations
- This notebook simulates application-level logging locally; it does not deploy a FastAPI service or connect Prometheus/Grafana.
- The metrics are parsed from log text, so changing log-message wording can break the parsing logic.
- A real service should use structured logs and explicit counters/histograms, plus log rotation, access control, and alerting.
- The Kidney Disease dataset and model are for educational demonstration only, not clinical decision-making.

### Student activity
Run the notebook from top to bottom, open `logs/ml_monitor.log`, and answer: Which errors were intentionally triggered? How are success and failure rates calculated? Why are latency percentiles useful in addition to average latency?